In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

True

Comprobar modelos Groq disponibles:

In [18]:
import requests
import os

api_key = os.getenv("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)
models = [model.get('id') for model in response .json().get('data')]
print(models)
                                          


['whisper-large-v3', 'allam-2-7b', 'openai/gpt-oss-120b', 'meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m', 'qwen/qwen3-32b', 'canopylabs/orpheus-arabic-saudi', 'openai/gpt-oss-20b', 'openai/gpt-oss-safeguard-20b', 'canopylabs/orpheus-v1-english', 'groq/compound', 'whisper-large-v3-turbo', 'llama-3.3-70b-versatile', 'groq/compound-mini', 'meta-llama/llama-4-scout-17b-16e-instruct', 'llama-3.1-8b-instant']


Comprobar conexión con Groq:

In [20]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

def llm_query(prompt, client):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [21]:
llm_query('Hello world!', openai_client)

"Hello. It's nice to meet you. Is there something I can help you with, or would you like to chat?"

Extracción de datos FAQ:

In [22]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [25]:
url_prefix="https://datatalks.club/faq"
response = requests.get(f"{url_prefix}/json/llm-zoomcamp.json")
faq_llm = response.json()

faq_llm

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '489dd1c9d9',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
  'answer':

In [26]:
faq_llm[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

Construir dataset de documentos:

In [29]:
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

Test *minsearch* library:

In [30]:
from minsearch import Index


index=Index(
    text_fields=['section','question','answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [31]:
question="How can I join the course?"

index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have r

In [ ]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

search("How will the homework be submitted?")

[{'id': '9a2e2d2008',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'How is my capstone project going to be evaluated?',
  'answer': 'Each submitted project will be evaluated by three randomly assigned students who have also submitted the project.\n\nYou will also be responsible for grading the projects from three fellow students yourself. Please be aware that not complying with this rule implies you may fail to achieve the Certificate at the end of the course.\n\nThe final grade you receive will be the median score of the grades from the peer reviewers. The peer review criteria for evaluation must follow the guidelines defined here (TBA for link).'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2025.'},
 {'id': 'eae0fb50aa',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'When and how will we be assign

In [41]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()


def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [44]:
question="How will the capstone project be evaluated?"
results=search(question)
prompt = build_prompt(question, results)
prompt

'Question:\nHow will the capstone project be evaluated?\n\nContext:\nCapstone Project\nQ: How is my capstone project going to be evaluated?\nA: Each submitted project will be evaluated by three randomly assigned students who have also submitted the project.\n\nYou will also be responsible for grading the projects from three fellow students yourself. Please be aware that not complying with this rule implies you may fail to achieve the Certificate at the end of the course.\n\nThe final grade you receive will be the median score of the grades from the peer reviewers. The peer review criteria for evaluation must follow the guidelines defined here (TBA for link).\n\nCapstone Project\nQ: Is it a group project?\nA: No, the capstone is a solo project.\n\nCapstone Project\nQ: Does the competition count as the capstone?\nA: No, it does not. You can participate in the [math-kaggle-llm-competition](https://datatalks-club.slack.com/archives/C0791HB4A58) as a group if you want to form teams; but the

In [45]:
llm_query(prompt, openai_client)

"The capstone project will be evaluated through a peer review process. Here's how it works:\n\n1. Each submitted project will be evaluated by three randomly assigned students who have also submitted the project.\n2. You will also be responsible for grading the projects from three fellow students yourself. Failure to comply with this rule may result in not achieving the Certificate at the end of the course.\n3. The final grade you receive will be the median score of the grades from the peer reviewers.\n4. The peer review criteria for evaluation must follow the guidelines defined (link to be provided).\n\nNote that the project is an individual attempt, and you will be assigned three projects to review and grade after the submission deadline has passed."